# Outil de débogage interactif des événements de la LNH

Ce notebook permet d'explorer interactivement les événements play-by-play
d'un match de la LNH et de visualiser leurs coordonnées sur une patinoire.

In [1]:
%matplotlib inline # Afficher les graphiques Matplotlib directement dans le notebook

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
rink_path = Path("../figures/nhl_rink.png")
rink_image = mpimg.imread(rink_path)

In [3]:
def display_event(event, rink_image):
    """
    Afficher un événement play-by-play sur la patinoire.

    Si l'événement possède des coordonnées, sa position est dessinée
    sur la patinoire. Sinon, les données de l'événement sont affichées.
    """

    details = event.get("details", {})

    x = details.get("xCoord")
    y = details.get("yCoord")

    # Afficher les informations principales
    event_type = event.get("typeDescKey", "inconnu")
    period = event.get("periodDescriptor", {}).get("number", "?")
    time = event.get("timeInPeriod", "?")

    print(f"Type : {event_type}")
    print(f"Période : {period}")
    print(f"Temps : {time}")

    # Certains événements ne possèdent pas de coordonnées
    if x is None or y is None:
        print("Coordonnées indisponibles.")
        print()
        print(event)
        return

    print(f"xCoord : {x}")
    print(f"yCoord : {y}")

    # Dessiner l'événement sur la patinoire
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.imshow(
        rink_image,
        extent=[-100, 100, -42.5, 42.5]
    )

    ax.scatter(x, y, s=100)

    ax.set_xlim(-100, 100)
    ax.set_ylim(-42.5, 42.5)

    ax.set_xlabel("Coordonnée x (pieds)")
    ax.set_ylabel("Coordonnée y (pieds)")

    ax.set_title(
        f"{event_type} — Période {period} — {time}"
    )

    plt.show()

## 2. Sélection interactive de la saison et du match

L'outil permet de sélectionner une saison, le type de match
(saison régulière ou séries éliminatoires), un match particulier,
puis de parcourir tous les événements play-by-play.

In [4]:
DATA_ROOT = Path("../data/raw")


def season_label(season_id):
    """Convertir 20232024 en 2023-2024."""
    season_id = str(season_id)
    return f"{season_id[:4]}-{season_id[4:]}"


# Détecter automatiquement les saisons disponibles localement
season_ids = sorted(
    int(folder.name)
    for folder in DATA_ROOT.iterdir()
    if folder.is_dir() and folder.name.isdigit()
)


season_dropdown = widgets.Dropdown(
    options=[
        (season_label(season_id), season_id)
        for season_id in season_ids
    ],
    value=season_ids[-1],
    description="Saison :"
)


game_type_toggle = widgets.ToggleButtons(
    options=[
        ("Saison régulière", "regular"),
        ("Séries éliminatoires", "playoffs"),
    ],
    value="regular",
    description="Type :"
)


game_dropdown = widgets.Dropdown(
    description="Match :"
)


event_slider_full = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Événement :",
    continuous_update=False
)


interactive_output = widgets.Output()


# État courant de l'outil
state = {
    "plays": [],
    "game_id": None,
}


def update_game_list(change=None):
    """Mettre à jour la liste des matchs disponibles."""

    season_id = season_dropdown.value
    game_type = game_type_toggle.value

    game_dir = (
            DATA_ROOT
            / str(season_id)
            / game_type
    )

    game_files = sorted(game_dir.glob("*.json"))

    game_dropdown.options = [
        (file.stem, file)
        for file in game_files
    ]

    if game_files:
        game_dropdown.value = game_files[0]
    else:
        state["plays"] = []
        state["game_id"] = None

        with interactive_output:
            clear_output(wait=True)
            print("Aucun match disponible.")


def load_selected_game(change=None):
    """Charger le match sélectionné."""

    game_path = game_dropdown.value

    if game_path is None:
        return

    with open(game_path, "r", encoding="utf-8") as fichier:
        data = json.load(fichier)

    state["plays"] = data.get("plays", [])
    state["game_id"] = data.get("id", game_path.stem)

    if state["plays"]:
        event_slider_full.max = len(state["plays"]) - 1
        event_slider_full.value = 0
        display_selected_event()
    else:
        event_slider_full.max = 0

        with interactive_output:
            clear_output(wait=True)
            print("Aucun événement disponible pour ce match.")


def display_selected_event(change=None):
    """Afficher l'événement sélectionné."""

    if not state["plays"]:
        return

    event_index = event_slider_full.value
    event = state["plays"][event_index]

    with interactive_output:
        clear_output(wait=True)

        print(f"GAME_ID : {state['game_id']}")
        print(
            f"Événement {event_index + 1} / "
            f"{len(state['plays'])}"
        )
        print()

        display_event(event, rink_image)


# Observer les changements
season_dropdown.observe(
    update_game_list,
    names="value"
)

game_type_toggle.observe(
    update_game_list,
    names="value"
)

game_dropdown.observe(
    load_selected_game,
    names="value"
)

event_slider_full.observe(
    display_selected_event,
    names="value"
)


# Afficher l'interface
display(
    widgets.VBox([
        season_dropdown,
        game_type_toggle,
        game_dropdown,
        event_slider_full,
    ]),
    interactive_output
)


# Initialiser l'outil
update_game_list()

Output()

## Résumé

Cet outil interactif permet de sélectionner une saison, le type de match,
un match particulier et de parcourir tous les événements play-by-play.

Lorsqu'un événement possède des coordonnées, sa position est affichée sur
la patinoire. Lorsqu'aucune coordonnée n'est disponible, les données brutes
de l'événement sont affichées directement.